In [51]:
import py_vncorenlp
import os
from nltk import word_tokenize

In [52]:
os.environ["JAVA_HOME"] = r"C:\Program Files\Java\jdk-26" 
try:
    path # type: ignore
except:
    path=os.getcwd()
    model = py_vncorenlp.VnCoreNLP(save_dir=f"{path}\\VnCoreNLP", annotators=["wseg", "pos", "parse"]) # type: ignore


In [53]:
raw_texts= []
with open(f"{path}/inputs/raw_contracts.txt", "r", encoding="utf-8") as f:
    for line in f:
        raw_texts.append(line.strip())

In [54]:
def has_SV_structure(text, is_left=False):
    annotated_texts=model.annotate_text(text)[0]
    if is_left:
        for item in annotated_texts:
            if item["wordForm"].lower() in ["nếu"]:
                return False
    has_S = False
    for item in annotated_texts:
        if item["depLabel"] == 'sub':
            has_S = True

        if item["depLabel"] == 'root' and item["posTag"] in ["V"]:
            return has_S
    return False        
        
def split_clause(sentence:str):
    """input must be one sentence only"""
    tokens = word_tokenize(sentence)
    if not tokens:
        return []
    connecting_words = ["và", ","]
    if tokens[0] and not tokens[0][0].isalpha():
        if len(tokens) > 1:
            return split_clause(" ".join(tokens[1:]))
        else:
            return [sentence]
    for i,token in enumerate(tokens):

        if token in connecting_words:
            left, right = tokens[:i], tokens[i+1:]
            left_str, right_str = " ".join(left), " ".join(right)
            if left_str == "":
                return split_clause(right_str)
            if right_str == "":
                return split_clause(left_str)
            if has_SV_structure(left_str, True) and has_SV_structure(right_str):
                return [left_str] + split_clause(right_str)
    return [sentence]+["\n"]
text="Bên B sẽ thanh toán toàn bộ tiền thuê trước ngày 5 hàng tháng, và nếu thanh toán trễ hạn, mức phạt 1% mỗi ngày sẽ được áp dụng."
text=model.word_segment(text)
text=text[0]
split_clause(text)



['Bên B sẽ thanh_toán toàn_bộ tiền thuê trước ngày 5 hàng tháng , và nếu thanh_toán trễ hạn , mức phạt 1% mỗi ngày sẽ được áp_dụng .',
 '\n']

In [65]:
import re

def is_useful(sentence: str) -> bool:
    """
    Kiểm tra xem câu có chứa mệnh đề pháp lý hữu ích hay không.
    """
    text = sentence.strip()
    
    # 1. Lọc dựa trên độ dài: Mệnh đề pháp lý thường không quá ngắn.
    # Các dòng có dưới 3 từ thường là chữ ký, hoặc lỗi ngắt dòng.
    tokens = text.split()
    if len(tokens) < 3:
        return False

    # 2. Lọc cấu trúc đánh dấu điều khoản (Headers)
    # Dùng Regular Expression để nhận diện các dạng: "Điều 1.", "CHƯƠNG 2", "1.1."
    header_pattern = r'^(điều\s+\d+|chương\s+[ivxlc]+|\d+\.\d+|\d+\.)\b'
    if re.match(header_pattern, text.lower()):
        # Nếu dòng CHỈ chứa tiêu đề (ví dụ: "Điều 1.") thì bỏ qua.
        # Nếu có nội dung theo sau (ví dụ: "Điều 1. Bên B sẽ thanh toán..."), ta vẫn giữ lại 
        # nhưng lý tưởng nhất là bạn nên cắt bỏ phần "Điều 1." ở khâu tiền xử lý trước đó.
        if len(tokens) < 5: 
            return False
        
    key_value_pattern = r'^([a-zA-ZđĐ]\s*\)|[-+*])\s*[^:]+\s*:\s*.*$'
    if re.match(key_value_pattern, text):
        # Nếu câu này quá dài (VD > 20 từ), có thể nó là một mệnh đề hoàn chỉnh nằm sau dấu ":" 
        # (VD: "b) Trường hợp bất khả kháng: Bên B sẽ không phải chịu phạt...").
        # Do đó, chỉ cắt bỏ nếu nó là một cụm Key-Value ngắn.
        if len(tokens) < 15:
            return False

    # 3. Lọc cụm từ rập khuôn (Boilerplate)
    # Các thành phần thủ tục không mang lại giá trị trích xuất "ai làm gì"
    boilerplate_phrases = [
        "cộng hòa xã hội chủ nghĩa việt nam",
        "độc lập - tự do - hạnh phúc",
        "ký và ghi rõ họ tên",
        "ký",
        "đại diện bên a",
        "đại diện bên b",
        "hôm nay, ngày",
        "căn cứ luật",
        
    ]
    text_lower = text.lower()
    for phrase in boilerplate_phrases:
        if phrase in text_lower and len(tokens):
            return False

    # 4. Lọc dựa trên từ loại (POS Tagging)
    # Từ góc độ ngôn ngữ học, một mệnh đề thao tác (quy định hành động/trạng thái) 
    # trong hợp đồng bắt buộc phải có ít nhất một Động từ (V) hoặc Danh từ (N).
    try:
        # Tận dụng biến model VnCoreNLP ở global scope
        annotated_texts = model.annotate_text(text)[0]
        has_verb_or_noun = any(item["posTag"] in ["V", "N", "Np"] for item in annotated_texts)
        has_root_verb=any(item["posTag"] in ["V"] and item["depLabel"]=="root" for item in annotated_texts)
        if not has_SV_structure(text):
            return False
        if not has_root_verb:
            return False
        if not has_verb_or_noun:
            return False
    except Exception as e:
        # Bỏ qua bước này nếu gặp lỗi (ví dụ chuỗi chứa ký tự đặc biệt khiến parser lỗi)
        pass

    return True

In [66]:
import re

def clean_prefix(sentence: str) -> str:
    """
    Loại bỏ các ký tự đánh dấu danh sách ở đầu câu.
    Hỗ trợ các định dạng: "a)", "a )", "1.", "1.1.", "-", "+", "*"
    """
    text = sentence.strip()
    
    # Biểu thức chính quy (Regex) giải thích:
    # ^                 : Bắt đầu chuỗi
    # [a-zA-ZđĐ]\s*[\)\.] : Một chữ cái (bao gồm đ, Đ tiếng Việt), theo sau là khoảng trắng (tùy chọn) và dấu ngoặc đóng ")" hoặc dấu chấm "."
    # |                 : HOẶC
    # [-+*]             : Các ký tự gạch đầu dòng
    # |                 : HOẶC
    # \d+(?:\.\d+)*\s*[\.\)] : Số đếm (1., 2., 1.1.) theo sau là dấu chấm hoặc ngoặc đóng
    # \s* : Bỏ qua các khoảng trắng dư thừa sau marker
    
    pattern = r'^([a-zA-ZđĐ]\s*[\)\.]|[-+*]|\d+(?:\.\d+)*\s*[\.\)])\s*'
    
    # Thay thế phần marker tìm được bằng chuỗi rỗng
    cleaned_text = re.sub(pattern, '', text)
    
    return cleaned_text

In [67]:
sentences: list[str] = []
with open(f"{path}/inputs/raw_contracts.txt", encoding="utf-8") as f:
    for line in f.readlines():
        if is_useful(line):
            # 1. Làm sạch tiền tố (loại bỏ "a )", "-", "1.",...)
            cleaned_line = clean_prefix(line)
            
            # 2. Đưa vào VnCoreNLP để cắt từ
            segmented = model.word_segment(cleaned_line)
            sentences.extend(segmented)

# Loại bỏ dấu gạch dưới do bộ word segmenter tạo ra
for i in range(len(sentences)):
    sentences[i] = sentences[i].replace("_", " ")
# Loại bỏ dấu gạch dưới do bộ word segmenter tạo ra
for i in range(len(sentences)):
    sentences[i] = sentences[i].replace("_", " ")


In [68]:
total_clauses=[]
for sentence in sentences:
    clauses= split_clause(sentence)
    total_clauses.extend(clauses)
total_clauses

['Bên A cung cấp dịch vụ phát triển Hệ thống Quản lý Bệnh viện Điện tử ( HIS ) phiên bản 3.0 .',
 '\n',
 'Phạm vi : phân tích , thiết kế , phát triển , triển khai , đào tạo và bảo trì 24 tháng .',
 '\n',
 'Bên A cam kết hệ thống đáp ứng tiêu chuẩn bảo mật dữ liệu y tế theo Bộ Y tế .',
 '\n',
 'Bên A báo cáo tiến độ mỗi 02 tuần và họp review hàng tháng .',
 '\n',
 'Chậm tiến độ quá 15 ngày , Bên B có quyền yêu cầu tăng nhân lực hoặc phạt theo Điều 7 .',
 '\n',
 'Bên A không được sử dụng mã nguồn tuỳ chỉnh cho bên thứ ba .',
 '\n',
 'Bên B có quyền yêu cầu chuyển giao toàn bộ tài liệu kỹ thuật .',
 '\n',
 'Bên A cam kết bảo mật dữ liệu bệnh nhân , hồ sơ y tế và thông tin kinh doanh .',
 '\n',
 'Nghiêm cấm Bên A truy cập , sao chép hoặc chia sẻ dữ liệu bệnh nhân cho bên thứ ba .',
 '\n',
 'Nghĩa vụ bảo mật kéo dài vô thời hạn đối với dữ liệu y tế .',
 '\n',
 'Vi phạm bảo mật : bồi thường tối thiểu 500.000.000 VNĐ .',
 '\n',
 'Bảo hành 24 tháng kể từ ngày nghiệm thu .',
 '\n',
 'Hỗ trợ kỹ 

In [69]:
with open(f"{path}/output/clauses.txt", "w", encoding="utf-8") as f:
    for clause in total_clauses:
        f.write(clause+"\n")
